# 이터널 리턴 유저 행동 데이터 전처리
> **목적**: 유저별 플레이 로그를 바탕으로 리텐션(잔존율) 분석을 위한 핵심 지표를 생성하고 데이터 무결성을 확보합니다.

### 주요 작업 내용:
1. **데이터 로드**: 약 수백만 건의 플레이 로그 및 유저 메타 데이터 수집
2. **유저 정의**: 계정 레벨 및 플레이 횟수를 기준으로 '초보 유저' 세그먼트 정의
3. **피처 엔지니어링**: 봇 노출도(Bot Exposure), MMR 변동폭, 평균 킬 수 등 파생 변수 생성
4. **리텐션 라벨링**: 특정 기간 내 재방문 여부를 기준으로 `is_retained` 타겟 변수 생성

# User Data Preprocessing

## STEP 0. 환경 설정
- 라이브러리 import
- 경고 제거 및 한글 폰트 설정

In [ ]:
# STEP 0. 환경 설정

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

# 시각화 (필요 시)
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 (선택)
plt.rcParams['font.family'] = 'Malgun Gothic'

## STEP 1. 데이터 로드
- 원본 데이터 불러오기
- 데이터 구조 확인

In [ ]:
# STEP 1. 데이터 로드

df = pd.read_csv('data/raw/game_data.csv', encoding='utf-8')

print(df.shape)
df.head()

## STEP 2. 컬럼 정리 및 타입 변환
- 날짜형 변환
- 숫자형 변환
- 불필요 컬럼 제거

In [ ]:
# STEP 2. 컬럼 정리 및 타입 변환

# datetime 변환
df['gameDate'] = pd.to_datetime(df['gameDate'])

# 숫자형 변환
numeric_cols = ['kill', 'playTime', 'mmr']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# 필요 없는 컬럼 제거
drop_cols = ['unnecessary_col1', 'unnecessary_col2']
df = df.drop(columns=drop_cols, errors='ignore')

## STEP 3. 결측치 처리
- 핵심 컬럼 결측 제거
- 나머지 값 대체 처리

In [ ]:
# STEP 3. 결측치 처리

# 핵심 컬럼 결측 제거
df = df.dropna(subset=['userID', 'playTime'])

# 나머지는 0으로 대체
df['kill'] = df['kill'].fillna(0)

## STEP 4. 이상치 처리
- 비정상 플레이 데이터 제거
- 극단값 필터링

In [ ]:
# STEP 4. 이상치 처리

# 플레이타임 0 제거
df = df[df['playTime'] > 0]

# 극단값 제거 (상위 1%)
upper = df['playTime'].quantile(0.99)
df = df[df['playTime'] <= upper]

## STEP 5. 초보 유저 정의
- 레벨, 킬 수, MMR 기준 정의
- 초보 유저 Flag 생성

In [ ]:
# STEP 5. 초보 유저 정의

df['is_novice'] = (
    (df['level'] <= 20) &
    (df['kill'] <= 3) &
    (df['mmr'] <= 1200) &
    (df['legend_item'] == 0)
).astype(int)

## STEP 6. 파생 변수 생성
- 봇 비율
- 실제 전투 지표
- 매칭 품질 지표

In [ ]:
# STEP 6. 파생 변수 생성

# 봇 비율
df['bot_exposure'] = df['bot_kill'] / (df['kill'] + 1)

# 실제 유저 킬
df['human_kill_score'] = df['kill'] - df['bot_kill']

# 매칭 품질 (예시)
df['match_quality'] = df['mmr'] / df['rank']

## STEP 7. 유저 단위 집계
- 유저별 플레이 패턴 요약
- 평균 및 횟수 기반 집계

In [ ]:
# STEP 7. 유저 단위 집계

user_df = df.groupby('userID').agg({
    'gameDate': 'count',
    'kill': 'mean',
    'playTime': 'mean',
    'bot_exposure': 'mean'
}).reset_index()

user_df.rename(columns={'gameDate': 'total_games'}, inplace=True)

## STEP 8. 리텐션 변수 생성
- 잔존 유저 정의
- 재방문 여부 Flag 생성

In [ ]:
# STEP 8. 리텐션 변수 생성

user_df['is_retained'] = (user_df['total_games'] >= 3).astype(int)

## STEP 9. 데이터 저장
- 최종 유저 데이터셋 저장

In [ ]:
# STEP 9. 데이터 저장

user_df.to_csv('data/processed/user_final.csv', index=False)